# Level 2: accuracy and latency scorecard

Runs a 25-question set through the Conversation API with a 5 QPM throttle. TIMEOUT, NO_ANSWER, and ERROR are FAIL. Writes `scorecard_runs` for the Test Lab app.


## Bind config


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("telemetry_rows", "10000000", "Telemetry rows (10000000 or 50000000)")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Score A vs B

Uses the golden JSON. Eval-runs API is Beta; Conversation API is the fallback used here.


In [ ]:
import json
import time
from shared.lib.genie_client import GenieClientError

ids = spark.table(cfg.table("agent_ids_l2")).collect()[0]
client = GenieClient(warehouse_id=cfg.warehouse_id)
golden = json.loads((root / "level-2-performance-lab" / "benchmarks" / "golden.json").read_text())
questions = golden["questions"]

def ask(space_id, question):
    t0 = time.time()
    try:
        started = client.start_conversation(space_id, question)
        conversation_id = started.get("conversation_id") or (started.get("conversation") or {}).get("id")
        message = started.get("message") or {}
        message_id = message.get("id") or started.get("message_id")
        done = client.wait_for_message(space_id, conversation_id, message_id)
        status = (done.get("status") or "UNKNOWN").upper()
        verdict = "PASS" if status == "COMPLETED" else "FAIL"
        return verdict, status, time.time() - t0
    except GenieClientError as exc:
        return "FAIL", str(exc)[:80], time.time() - t0

rows = []
for item in questions:
    q = item["question"]
    for label, sid in [("A", ids["agent_a_id"]), ("B", ids["agent_b_id"])]:
        verdict, status, latency = ask(sid, q)
        rows.append((item["id"], item.get("category", "na"), label, q, verdict, status, float(latency)))
        print(label, verdict, round(latency, 1), q)
        time.sleep(13)

spark.createDataFrame(rows, ["qid", "category", "agent", "question", "verdict", "status", "latency_s"]).write.mode(
    "overwrite"
).option("overwriteSchema", "true").saveAsTable(cfg.table("scorecard_runs"))
print("wrote", cfg.table("scorecard_runs"))
spark.sql(
    f"""SELECT agent, COUNT_IF(verdict='PASS') AS passes, COUNT(*) AS n, approx_percentile(latency_s, 0.5) AS p50, approx_percentile(latency_s, 0.95) AS p95
    FROM {cfg.table('scorecard_runs')} GROUP BY agent"""
).show()
